In [2]:
import torch

print("PyTorch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU disponível: True
GPU: Tesla T4


In [3]:
!pip install -q "transformers>=4.45,<5" datasets accelerate scikit-learn pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
from google.colab import files

arquivos = files.upload()

Saving avaliacoes_tratadas.csv to avaliacoes_tratadas.csv


In [5]:
import pandas as pd

df = pd.read_csv("avaliacoes_tratadas.csv")

print(df.shape)
print(df["sentimento"].value_counts())
print(df.head())

(16293, 3)
sentimento
1    15207
0     1086
Name: count, dtype: int64
                                               texto  avaliacao  sentimento
0  Comida maravilhosa, serviço excelente! lu\nlug...          5           1
1  Lugar excelente para ir com a família, super a...          4           1
2  Uma experiencia incrível, não deixe de visitar...          5           1
3  Melhor lugar que eu já fui! Recomendo fortemen...          5           1
4  Português/English \nObrigada Alex Atala e equi...          5           1


In [6]:
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Datasets: 4.8.5
Accelerate: 1.15.0
GPU disponível: True
GPU: Tesla T4


In [7]:
from sklearn.model_selection import train_test_split

X = df["texto"]
y = df["sentimento"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_treino, y_treino,
    test_size=0.1,
    random_state=42,
    stratify=y_treino
)

print("Treinamento:", len(X_treino))
print("Validação:", len(X_validacao))
print("Teste:", len(X_teste))

Treinamento: 11730
Validação: 1304
Teste: 3259


In [8]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

nome_modelo = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(nome_modelo)

def preparar_dataset(textos, sentimentos):
    dados = pd.DataFrame({
        "texto": textos.tolist(),
        "labels": sentimentos.tolist()
    })

    return Dataset.from_pandas(dados)

treino = preparar_dataset(X_treino, y_treino)
validacao = preparar_dataset(X_validacao, y_validacao)
teste = preparar_dataset(X_teste, y_teste)

def tokenizar(exemplos):
    return tokenizer(
        exemplos["texto"],
        truncation=True,
        max_length=128
    )

treino = treino.map(tokenizar, batched=True, remove_columns=["texto"])
validacao = validacao.map(tokenizar, batched=True, remove_columns=["texto"])
teste = teste.map(tokenizar, batched=True, remove_columns=["texto"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenização concluída!")
print("Treinamento:", len(treino))
print("Validação:", len(validacao))
print("Teste:", len(teste))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/11730 [00:00<?, ? examples/s]

Map:   0%|          | 0/1304 [00:00<?, ? examples/s]

Map:   0%|          | 0/3259 [00:00<?, ? examples/s]

Tokenização concluída!
Treinamento: 11730
Validação: 1304
Teste: 3259


In [9]:
from transformers import AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(
    nome_modelo,
    num_labels=2,
    id2label={0: "Negativa", 1: "Positiva"},
    label2id={"Negativa": 0, "Positiva": 1}
)

print("BERTimbau carregado com sucesso!")

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTimbau carregado com sucesso!


In [10]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer, TrainingArguments

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_treino
)

pesos = torch.tensor(pesos, dtype=torch.float32)

def calcular_metricas(resultado):
    previsoes = np.argmax(resultado.predictions, axis=1)
    reais = resultado.label_ids

    precisao, recall, f1, _ = precision_recall_fscore_support(
        reais, previsoes,
        labels=[0, 1],
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(reais, previsoes),
        "precision_neg": precisao[0],
        "recall_neg": recall[0],
        "f1_neg": f1[0],
        "f1_pos": f1[1]
    }

class TrainerBalanceado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=pesos.to(outputs.logits.device)
        )

        loss = loss_fn(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss

configuracao = TrainingArguments(
    output_dir="resultados_bert",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_neg",
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=42
)

treinador = TrainerBalanceado(
    model=modelo,
    args=configuracao,
    train_dataset=treino,
    eval_dataset=validacao,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=calcular_metricas
)

print("Treinamento configurado!")
print("Pesos:", pesos)

Treinamento configurado!
Pesos: tensor([7.5000, 0.5357])


In [11]:
treinador.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Neg,Recall Neg,F1 Neg,F1 Pos
1,0.237500,0.168872,0.984663,0.885057,0.885057,0.885057,0.991783
2,0.100700,0.148445,0.984663,0.860215,0.919540,0.888889,0.991763


TrainOutput(global_step=1468, training_loss=0.2175820081695224, metrics={'train_runtime': 221.3019, 'train_samples_per_second': 106.009, 'train_steps_per_second': 6.633, 'total_flos': 1503233625921360.0, 'train_loss': 0.2175820081695224, 'epoch': 2.0})

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

resultado = treinador.predict(teste)

previsoes = resultado.predictions.argmax(axis=1)
reais = resultado.label_ids

print("\nRelatório de classificação:")
print(classification_report(
    reais,
    previsoes,
    target_names=["Negativa", "Positiva"]
))

print("\nMatriz de confusão:")
print(confusion_matrix(reais, previsoes))


Relatório de classificação:
              precision    recall  f1-score   support

    Negativa       0.80      0.86      0.83       217
    Positiva       0.99      0.98      0.99      3042

    accuracy                           0.98      3259
   macro avg       0.90      0.92      0.91      3259
weighted avg       0.98      0.98      0.98      3259


Matriz de confusão:
[[ 186   31]
 [  46 2996]]


In [13]:
treinador.save_model("FoodReviewAI_BERTimbau")
tokenizer.save_pretrained("FoodReviewAI_BERTimbau")

print("Modelo BERTimbau salvo com sucesso!")

Modelo BERTimbau salvo com sucesso!


In [14]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

shutil.copytree(
    "FoodReviewAI_BERTimbau",
    "/content/drive/MyDrive/FoodReviewAI_BERTimbau",
    dirs_exist_ok=True
)

print("Modelo copiado para o Google Drive!")

Mounted at /content/drive
Modelo copiado para o Google Drive!


In [15]:
from huggingface_hub import notebook_login

notebook_login()

In [16]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="FoodReviewAI_BERTimbau",
    repo_id="felipermtrindade/FoodReviewAI-BERTimbau",
    repo_type="model",
    commit_message="Upload fine-tuned BERTimbau sentiment classifier"
)

print("Modelo publicado com sucesso!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...RTimbau/model.safetensors:   0%|          |  556kB /  436MB            

  ...RTimbau/training_args.bin:   2%|1         |  96.0B / 5.84kB            

Modelo publicado com sucesso!
